# Homework Submission - Stage 05: Data Storage

This notebook completes the official starter with environment-driven paths, CSV/Parquet persistence, reload validation, and reusable suffix-based utilities.

In [1]:
from pathlib import Path
import datetime as dt
import os

import numpy as np
import pandas as pd
from dotenv import load_dotenv

from src.storage import read_df, write_df

ROOT = Path.cwd()
load_dotenv(ROOT / '.env')
RAW = ROOT / os.getenv('DATA_DIR_RAW', 'data/raw')
PROCESSED = ROOT / os.getenv('DATA_DIR_PROCESSED', 'data/processed')
RAW.mkdir(parents=True, exist_ok=True)
PROCESSED.mkdir(parents=True, exist_ok=True)
RUN_TS = dt.datetime.now().strftime('%Y%m%d-%H%M%S')
print('RAW:', RAW)
print('PROCESSED:', PROCESSED)

RAW: /Users/zhangyuang/Desktop/py_file/bootcamp/homework/homework05/data/raw
PROCESSED: /Users/zhangyuang/Desktop/py_file/bootcamp/homework/homework05/data/processed


## 1. Create a deterministic market DataFrame

In [2]:
rng = np.random.default_rng(5)
dates = pd.bdate_range('2025-01-02', periods=20)
returns = rng.normal(0.0005, 0.012, size=len(dates))
df = pd.DataFrame({
    'date': dates,
    'ticker': 'AAPL',
    'price': (150 * np.exp(np.cumsum(returns))).round(4),
    'volume': rng.integers(40_000_000, 90_000_000, size=len(dates), endpoint=False),
})
df.info()
display(df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   date    20 non-null     datetime64[ns]
 1   ticker  20 non-null     object        
 2   price   20 non-null     float64       
 3   volume  20 non-null     int64         
dtypes: datetime64[ns](1), float64(1), int64(1), object(1)
memory usage: 768.0+ bytes


,date,ticker,price,volume
0,2025-01-02,AAPL,148.6377,87954969
1,2025-01-03,AAPL,146.3674,67779805
2,2025-01-06,AAPL,146.0048,85175591
3,2025-01-07,AAPL,146.8167,53572580
4,2025-01-08,AAPL,148.9063,58075911


## 2. Save one CSV snapshot and one Parquet snapshot

In [3]:
csv_path = write_df(df, RAW / f'sample_{RUN_TS}.csv')
parquet_path = write_df(df, PROCESSED / f'sample_{RUN_TS}.parquet')
print('CSV:', csv_path)
print('Parquet:', parquet_path)

CSV: /Users/zhangyuang/Desktop/py_file/bootcamp/homework/homework05/data/raw/sample_20260827-104459.csv
Parquet: /Users/zhangyuang/Desktop/py_file/bootcamp/homework/homework05/data/processed/sample_20260827-104459.parquet


## 3. Reload and validate

In [4]:
df_csv = read_df(csv_path, parse_dates=['date'])
df_parquet = read_df(parquet_path)

def validate_loaded(original: pd.DataFrame, reloaded: pd.DataFrame) -> dict:
    required = ['date', 'ticker', 'price', 'volume']
    return {
        'shape_equal': original.shape == reloaded.shape,
        'required_columns_present': set(required).issubset(reloaded.columns),
        'date_is_datetime': pd.api.types.is_datetime64_any_dtype(reloaded['date']),
        'price_is_numeric': pd.api.types.is_numeric_dtype(reloaded['price']),
        'volume_is_integer': pd.api.types.is_integer_dtype(reloaded['volume']),
        'ticker_values_match': original['ticker'].equals(reloaded['ticker']),
        'prices_match': bool(np.allclose(original['price'], reloaded['price'], rtol=0, atol=1e-10)),
        'na_count': int(reloaded[required].isna().sum().sum()),
    }

validation = pd.DataFrame({
    'csv': validate_loaded(df, df_csv),
    'parquet': validate_loaded(df, df_parquet),
})
display(validation)
boolean_rows = validation.index != 'na_count'
assert validation.loc[boolean_rows].to_numpy().all()
assert (validation.loc['na_count'] == 0).all()

,csv,parquet
shape_equal,True,True
required_columns_present,True,True
date_is_datetime,True,True
price_is_numeric,True,True
volume_is_integer,True,True
ticker_values_match,True,True
prices_match,True,True
na_count,0,0


## 4. Utility behavior and error messages

In [5]:
from src.storage import detect_format

print('CSV route:', detect_format('example.csv'))
print('Parquet route:', detect_format('example.parquet'))
try:
    detect_format('example.xlsx')
except ValueError as exc:
    print('Expected unsupported-format message:', exc)

missing_demo = ROOT / 'data' / 'raw' / 'does_not_exist.csv'
try:
    read_df(missing_demo)
except FileNotFoundError as exc:
    print('Expected missing-file message:', exc)

CSV route: csv
Parquet route: parquet
Expected unsupported-format message: Unsupported storage format: .xlsx
Expected missing-file message: Data file not found: /Users/zhangyuang/Desktop/py_file/bootcamp/homework/homework05/data/raw/does_not_exist.csv


## 5. Assumptions and storage tradeoffs

- CSV must parse `date` explicitly on reload; Parquet preserves it.
- The exact numeric equality check is safe here because the generated prices are rounded before writing. Less controlled floating-point workflows may need a documented tolerance.
- Parquet is the processed analytical format, while CSV remains a portable raw interchange copy.
- The notebook proves round-trip correctness for this schema; a production utility would also support schema versions and checksums.